# RL-Chess-Engine — Draw-Cycle Experiments
**Experiment A** (compute scaling) + **Experiment B** (fixed-compute self-play signal ablation)  
Results auto-written to `docs/experiment-results.md` and committed back to GitHub.

**Prerequisites (set before running):**
- Kaggle notebook: Runtime → GPU T4 x2 (or any GPU)
- Kaggle notebook: Settings → Internet → On
- Kaggle Secrets: `GITHUB_TOKEN` (fine-grained PAT with repo write access)

## Step 1 — Confirm GPU

In [ ]:
!nvidia-smi -L
import torch
cuda_ok = torch.cuda.is_available()
print('CUDA:', cuda_ok)
assert cuda_ok, 'No GPU detected — enable GPU in notebook settings before continuing.'

## Step 2 — Authenticate and clone the repo

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['GH_TOKEN'] = UserSecretsClient().get_secret('GITHUB_TOKEN')
!git clone https://$GH_TOKEN@github.com/Danny-397/RL-Chess-Engine.git
%cd RL-Chess-Engine
!git config user.email 'dlichtenberger91@gmail.com'
!git config user.name  'Danny-397'
print('Cloned and configured.')

## Step 3 — Install the one missing dependency
*(Do NOT reinstall torch — Kaggle's build is already CUDA-enabled.)*

In [ ]:
!pip install -q 'python-chess>=1.11'
import chess
print('python-chess:', chess.__version__)

## Step 4 — Quick pipeline validation (~5 min, CPU, numbers meaningless)
Catches import errors or broken wiring before spending GPU time.

In [ ]:
!python run_experiment.py --quick

## Step 5 — Real experiments (Experiment A + B on GPU)
**Resumable** — if the session drops, re-run this cell; completed stages are skipped.
**Budget cap:** ~6 GPU-hours. Default flags run well within that.

In [ ]:
!python run_experiment.py --device cuda
# Optional bigger run (uncomment if budget allows):
# !python run_experiment.py --device cuda --scaled-iters 40 --scaled-games 100 --scaled-sims 200

## Step 6 — Inspect results and charts

In [ ]:
!cat docs/experiment-results.md

In [ ]:
import os
from IPython.display import Image, display
for name in ['term_baseline','term_scaled','term_assisted',
             'progress_baseline','progress_scaled','progress_assisted']:
    path = f'assets/{name}.png'
    if os.path.exists(path):
        print(f'\n--- {name} ---')
        display(Image(path))
    else:
        print(f'(chart not found: {path})')

## Step 7 — Fill §5.3 of `docs/technical-report.md`
Reads the three `experiment_out/<stage>.result.json` files and replaces every
`⟨FILL⟩` placeholder in §5.3 with real numbers. Everything outside §5.3 is untouched.

In [ ]:
import json, re

def load_result(stage):
    with open(f'experiment_out/{stage}.result.json', encoding='utf-8') as f:
        return json.load(f)

b = load_result('baseline')
s = load_result('scaled')
a = load_result('assisted')

print('Loaded results:')
for m in (b, s, a):
    print(f"  {m['name']:>12}: non-decisive={m['final_non_decisive_rate']:.0%}  "
          f"Elo={m['elo_vs_random']:+.0f}  games={m['total_self_play_games']}  "
          f"material_w={m.get('material_weight', 0.0):.2f}")

# Build replacement rows
# Experiment A (cols: Run | Self-play games | Non-decisive rate | Elo vs random)
a_base = f"| baseline | {b['total_self_play_games']} | {b['final_non_decisive_rate']:.0%} | {b['elo_vs_random']:+.0f} |"
a_scal = f"| scaled | {s['total_self_play_games']} | {s['final_non_decisive_rate']:.0%} | {s['elo_vs_random']:+.0f} |"

# Experiment B (cols: Run | Self-play games | Material w | Non-decisive rate | Elo vs random)
b_base = f"| baseline | {b['total_self_play_games']} | 0.00 | {b['final_non_decisive_rate']:.0%} | {b['elo_vs_random']:+.0f} |"
b_asst = (f"| assisted | {a['total_self_play_games']} | {a.get('material_weight',0.0):.2f} "
          f"| {a['final_non_decisive_rate']:.0%} | {a['elo_vs_random']:+.0f} |")

with open('docs/technical-report.md', encoding='utf-8') as f:
    report = f.read()

# 1. Inline game-count mentions in the Experiment A paragraph
report = re.sub(
    r'baseline \\(~⟨FILL⟩ games\\) and a\\s+larger scaled run \\(~⟨FILL⟩ games on GPU\\)',
    f"baseline (~{b['total_self_play_games']} games) and a\nlarger scaled run (~{s['total_self_play_games']} games on GPU)",
    report
)

# 2. Experiment A table rows
report = report.replace(
    '| baseline | ⟨FILL⟩ | ⟨FILL⟩ | ⟨FILL⟩ |\n| scaled | ⟨FILL⟩ | ⟨FILL⟩ | ⟨FILL⟩ |',
    a_base + '\n' + a_scal
)

# 3. Experiment B table rows
report = report.replace(
    '| baseline | ⟨FILL⟩ | 0.00 | ⟨FILL⟩ | ⟨FILL⟩ |\n| assisted | ⟨FILL⟩ | ⟨FILL⟩ | ⟨FILL⟩ | ⟨FILL⟩ |',
    b_base + '\n' + b_asst
)

with open('docs/technical-report.md', 'w', encoding='utf-8') as f:
    f.write(report)

# Verify
lines = report.split('\n')
sec_start = next((i for i, l in enumerate(lines) if '5.3' in l), 0)
sec_end   = next((i for i, l in enumerate(lines) if i > sec_start and l.startswith('### ')), sec_start + 80)
remaining = [(i, l) for i, l in enumerate(lines[sec_start:sec_end], sec_start) if '⟨FILL⟩' in l]

if remaining:
    print('\nWARNING: unfilled placeholders remain in §5.3:')
    for lineno, line in remaining:
        print(f'  line {lineno+1}: {line}')
else:
    print('\n✓ All ⟨FILL⟩ placeholders in §5.3 replaced.')

print('\n-- Updated §5.3 table rows --')
for line in lines[sec_start:sec_end]:
    if line.startswith('|'):
        print(line)

## Step 8 — Commit and push to GitHub

In [ ]:
# Stage only the specified files — do NOT add pgn/, logs/, experiment_out/
!git add docs/technical-report.md docs/experiment-results.md
!git add assets/term_baseline.png assets/term_scaled.png assets/term_assisted.png 2>/dev/null || true
!git add assets/progress_baseline.png assets/progress_scaled.png assets/progress_assisted.png 2>/dev/null || true

print('Files staged:')
!git diff --cached --name-only

!git commit -m 'Add draw-cycle experiment results (compute + signal ablation)'
!git push
print('\n✓ Pushed to GitHub.')

## Step 9 — Final summary

In [ ]:
import json, re

def load_result(stage):
    with open(f'experiment_out/{stage}.result.json', encoding='utf-8') as f:
        return json.load(f)

b = load_result('baseline')
s = load_result('scaled')
a = load_result('assisted')

print('=' * 65)
print('DRAW-CYCLE EXPERIMENT SUMMARY')
print('=' * 65)
print(f"{'Run':<12} {'Games':>8} {'Sims':>6} {'Mat-w':>6} {'Non-decisive':>14} {'Elo':>8}")
print('-' * 65)
for m in (b, s, a):
    print(f"{m['name']:<12} "
          f"{m['total_self_play_games']:>8} "
          f"{m['simulations']:>6} "
          f"{m.get('material_weight', 0.0):>6.2f} "
          f"{m['final_non_decisive_rate']:>13.0%} "
          f"{m['elo_vs_random']:>+8.0f}")
print('=' * 65)

with open('docs/experiment-results.md', encoding='utf-8') as f:
    results_md = f.read()

match = re.search(r'## Reading the two together\n(.+?)(?=\n#|\Z)', results_md, re.DOTALL)
if match:
    print('\nReading the two together:')
    print(match.group(1).strip())

print('\nArtifacts committed to GitHub:')
print('  docs/experiment-results.md')
print('  docs/technical-report.md  (§5.3 filled)')
print('  assets/term_{baseline,scaled,assisted}.png')
print('  assets/progress_{baseline,scaled,assisted}.png')